In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('..')

from Constants import db, model_db
from DBTypes import *
from ModelDBTypes import *
from DB_AdvancedStatements import *

In [ ]:
cursor = db.cursor()

pitcher_set = {0}
pitcher_first_months : list[tuple[DB_Model_PitcherStats, DB_PitcherStatcastMonth]] = []

years = [2021, 2022, 2023]
months = [4,5,6,7,8,9]

for year in years:
    for month in months:
        low_a_pitchers = Select_InnerJoin(DB_Model_PitcherStats, DB_PitcherStatcastMonth, cursor,
            '''SELECT mps.*, psm.*
            FROM Model_PitcherStats AS mps
            INNER JOIN PitcherStatcastMonth AS psm
                ON mps.mlbId=psm.mlbId
                AND mps.Year=psm.Year
                AND mps.Month=psm.Month
            WHERE mps.Year=?
            AND mps.Month=?
            AND mps.LevelId>=5
            AND mps.BF>30
            ''',
            (year, month)
        )
        
        for p in low_a_pitchers:
            mps, psm = p
            if not mps.mlbId in pitcher_set:
                pitcher_set.add(mps.mlbId)
                pitcher_first_months.append(p)

print(len(pitcher_first_months))

In [ ]:
model_cursor = model_db.cursor()

data : list[tuple[float, float]] = []

for mps, psm in pitcher_first_months:
    try:
        expected_war = DB_Output_PlayerWarAggregation.Select_From_DB(model_cursor,
            "WHERE mlbId=? AND model=1 AND isHitter=0 AND Year=? AND Month=?",
            (mps.mlbId, mps.Year, mps.Month))[0].war
        
        try:
            expected_war_2years = DB_Output_PlayerWarAggregation.Select_From_DB(model_cursor,
                "WHERE mlbId=? AND model=1 AND isHitter=0 AND Year=? AND Month=?",
                (mps.mlbId, mps.Year + 2, mps.Month))[0].war
        except:
            # Do last value
            expected_war_2years = DB_Output_PlayerWarAggregation.Select_From_DB(model_cursor,
                "WHERE mlbId=? AND model=1 AND isHitter=0 ORDER BY Year DESC, Month DESC",
                (mps.mlbId,))[0].war
            
        val = psm.StuffFastball
        if expected_war < 3 and val is not None:
            data.append((val, expected_war_2years - expected_war))
    except:
        pass # Not a prospect
    


In [ ]:
def sortFunction(d : tuple[float, float]):
    x, y = d
    return x

data.sort(key=sortFunction)

In [ ]:
sub_90 = []
_9095 = []
_95100 = []
_100105 = []
over_105 = []

def avg(xs : list[float]) -> float:
    if len(xs) == 0:
        return None
    
    return sum(xs) / len(xs)

for stuff, delta in data:
    if stuff < 90:
        sub_90.append(delta)
    elif stuff < 95:
        _9095.append(delta)
    elif stuff < 100:
        _95100.append(delta)
    elif stuff < 105:
        _100105.append(delta)
    else:
        over_105.append(delta)
        
print(f"{avg(sub_90):.3f} {len(sub_90)}")
print(f"{avg(_9095):.3f} {len(_9095)}")
print(f"{avg(_95100):.3f} {len(_95100)}")
print(f"{avg(_100105):.3f} {len(_100105)}")
print(f"{avg(over_105):.3f} {len(over_105)}")

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

xs = [d[0] for d in data]
ys = [d[1] for d in data]

x = np.array(xs).reshape(-1, 1)
y = np.array(ys).reshape(-1, 1)

lr = LinearRegression()
lr.fit(x, y)
print(f"Slope (coefficient): {lr.coef_[0][0]:.4f}")
print(f"Intercept: {lr.intercept_[0]:.4f}")
print(f"R² Score: {lr.score(x, y):.4f}")

In [ ]:
def GetWindowAvg(data : list[tuple[float, float]], value : float, delta : float) -> float:
    window_start = value - delta
    window_end = value + delta
    
    values = []
    
    for i in range(len(data)):
        x, y = data[i]
        if x > window_start and x < window_end:
            values.append(y)
        elif x >= window_end:
            break
        
    return avg(values)

In [ ]:
window_xs = range(80, 120)
window_ys = [GetWindowAvg(data, x, 3) for x in window_xs]

In [ ]:
neutral_xs = [80, 110]
neutral_ys = [0, 0]

In [ ]:
import matplotlib.pyplot as plt

plt.plot(xs, ys, 'ko')
plt.plot(window_xs, window_ys, 'r-')
plt.plot(neutral_xs, neutral_ys, 'g--')